In [13]:
import math
from neo4j import GraphDatabase

class MedikalyReranker:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def algorithm_2_reranking(self, d_can, eq_entities, top_n=5):
        # Clinical priority weights
        TYPE_WEIGHTS = {"sym": 3.0, "mic": 2.0, "ite": 2.0, "pro": 1.0}
        
        final_results = []
        with self.driver.session() as session:
            for d_name in d_can:
                total_path_score = 0
                weight_tie_breaker = 0
                match_count = 0
                evidence_list = []
                
                exists = session.run("MATCH (d {name: $n}) RETURN d LIMIT 1", n=d_name).single()
                if not exists: continue

                for e_name in eq_entities:
                    if d_name == e_name: continue
                    
                    # Query pulls the source/target weights we imported from your CSVs
                    query = """
                    MATCH p = shortestPath((d {name: $d_name})-[r*1..4]-(e {name: $e_name}))
                    WHERE any(lb IN labels(d) WHERE lb IN ['dis', 'Disease'])
                    RETURN length(p) AS distance, labels(e)[0] as type,
                           reduce(s = 0.0, rel IN relationships(p) | s + COALESCE(rel.source_weight, 0) + COALESCE(rel.target_weight, 0)) as csv_weight
                    LIMIT 1
                    """
                    res = session.run(query, d_name=d_name, e_name=e_name)
                    record = res.single()

                    if record:
                        dist = record['distance']
                        w_ti = TYPE_WEIGHTS.get(record['type'], 1.0)
                        
                        # Primary Score: Exponential Distance
                        dist_factor = 1.0 / (2 ** dist)
                        anchor_boost = 3.0 if "Procalcitonin" in e_name or "Leukocytosis" in e_name else 1.0
                        
                        total_path_score += (w_ti * anchor_boost * dist_factor)
                        
                        # Tie Breaker: Use the CSV weights (0.1638, etc.) 
                        # We divide by 1000 so it only affects the decimals
                        weight_tie_breaker += (record['csv_weight'] / 1000.0)
                        
                        match_count += 1
                        evidence_list.append(f"{e_name}(d:{dist})")

                if match_count > 0:
                    # Final Score = (Distance Synergy) + (CSV Evidence Strength)
                    final_score = (total_path_score * (match_count ** 2)) + weight_tie_breaker

                    final_results.append({
                        "disease": d_name, 
                        "score": final_score,
                        "evidence": evidence_list,
                        "hits": match_count
                    })

        return sorted(final_results, key=lambda x: x['score'], reverse=True)[:top_n]

if __name__ == "__main__":
    EQ = ["Hyperthermia (High Fever)", "Leukocytosis NOS", "Tachycardia NOS", "Chest Pain", "Procalcitonin high"]
    D_CAN = ["Severe sepsis", "Septic shock", "Salmonella septicemia", "Bacterial pneumonia NOS"]

    reranker = MedikalyReranker("neo4j://localhost:7687", "neo4j", "kg123456")
    try:
        top_diagnoses = reranker.algorithm_2_reranking(D_CAN, EQ, top_n=5)
        print("\n" + "="*65 + "\n🏥 MEDIKAL ALGORITHM 2: RESOLVED CLINICAL RANKING\n" + "="*65)
        for i, res in enumerate(top_diagnoses):
            print(f"RANK {i+1}: {res['disease']}")
            print(f"  > Resolved Score: {round(res['score'], 6)}")
            print(f"  > Evidence: {', '.join(res['evidence'])}")
            print("-" * 50)
    finally:
        reranker.close()


🏥 MEDIKAL ALGORITHM 2: RESOLVED CLINICAL RANKING
RANK 1: Salmonella septicemia
  > Resolved Score: 15.751009
  > Evidence: Leukocytosis NOS(d:2), Tachycardia NOS(d:2), Procalcitonin high(d:2)
--------------------------------------------------
RANK 2: Severe sepsis
  > Resolved Score: 15.751002
  > Evidence: Leukocytosis NOS(d:2), Tachycardia NOS(d:2), Procalcitonin high(d:2)
--------------------------------------------------
RANK 3: Septic shock
  > Resolved Score: 15.751002
  > Evidence: Leukocytosis NOS(d:2), Tachycardia NOS(d:2), Procalcitonin high(d:2)
--------------------------------------------------
RANK 4: Bacterial pneumonia NOS
  > Resolved Score: 10.688821
  > Evidence: Leukocytosis NOS(d:2), Tachycardia NOS(d:2), Procalcitonin high(d:4)
--------------------------------------------------
